# Bitcoin 15m Composite Quant Strategy
### 5-Signal Ensemble: EMA Trend · RSI Reversion · MACD Momentum · BB Squeeze · Volume Spike
---

In [ ]:
# Install dependencies (run once)
# pip install ccxt pandas numpy pandas-ta scikit-learn xgboost matplotlib seaborn joblib

In [ ]:
# Import the strategy module (save bitcoin_quant_strategy.py in the same folder)
from bitcoin_quant_strategy import *

# You can override CONFIG here if needed:
# CONFIG['limit'] = 3000
# CONFIG['timeframe'] = '5m'
print('Modules loaded OK')

## Step 1 — Fetch Data

In [ ]:
df_raw = fetch_ohlcv(CONFIG)
df_raw.tail()

## Step 2 — Compute Indicators (All 5 Strategies)

In [ ]:
df = compute_indicators(df_raw, CONFIG)
df[['close','ema_fast','ema_slow','sig_ema',
    'rsi','sig_rsi','macd_hist','sig_macd',
    'bb_width','sig_bb','vol_spike','sig_vol',
    'composite_score','rule_signal']].tail(10)

## Step 3 — Label Data

In [ ]:
df = label_data(df, CONFIG)
df['label'].value_counts().rename({0:'SHORT',1:'FLAT',2:'LONG'}).plot.bar(color=['#ef5350','#78909c','#26a69a'])

## Step 4 — Train Model

In [ ]:
X, y = build_feature_matrix(df)
model, scaler, xgb_model = train_model(X, y, CONFIG)

## Step 5 — Backtest (Rule-Based)

In [ ]:
trades_rule = backtest(df, CONFIG, use_ml=False)
trades_rule.head(10)

## Step 6 — Backtest (ML Model)

In [ ]:
trades_ml = backtest(df, CONFIG, use_ml=True, model=model, scaler=scaler, X=X)
trades_ml.head(10)

## Step 7 — Live Prediction (Latest Bar)

In [ ]:
pred = predict_latest(df, model, scaler, CONFIG)
import json
print(json.dumps(pred, indent=2, default=str))

## Step 8 — Dashboard & Feature Importance

In [ ]:
plot_strategy_dashboard(df, trades_rule, CONFIG)

In [ ]:
plot_feature_importance(xgb_model, X.columns.tolist())